# WSDM 2027 — Experiment 1: pilot extraction thật trên Kaggle

Notebook này chạy **5 argument units chẩn đoán đã chọn có chủ đích** trong điều kiện `human_segmented`. Đây là kiểm tra kỹ thuật và định tính, **không phải đánh giá accuracy không thiên lệch**.

Phương pháp được giữ nguyên từ repository:

$$S(A\rightarrow B\mid C)=Temporal\times G\times U\times D$$

Notebook tái sử dụng trực tiếp `wsdm/llm.py`, `wsdm/steering.py` và `wsdm/approved_detection_experiment.py` tại commit đã xác minh. Không đưa gold label, adjudication note hoặc gold evidence span vào prompt.

## Chuẩn bị trên Kaggle

1. Import notebook này vào Kaggle.
2. Trong **Settings**, bật **Internet** và chọn accelerator **GPU T4 x2** (notebook chỉ dùng GPU số 0; một T4 16 GB là đủ cho model Q4 4,7 GB). Không đổi sang P100 nếu chưa kiểm tra tương thích driver Ollama.
3. Tạo một Kaggle Dataset riêng tư chứa đúng file `model_inputs_human_segmented_5.csv` từ thư mục local:
   `wsdm/results/experiment1_real_llm_diagnostic_20261117T012052/`.
4. Chọn **Add Input** và attach dataset đó vào notebook. Không attach `gold_labels.csv`, annotation gốc hoặc evaluation manifest.
5. Chạy từng checkpoint theo thứ tự và xem output trước khi tiếp tục.


## Checkpoint 1 — Xác minh GPU

Chạy cell dưới đây trước tiên. Dừng nếu không có CUDA GPU hoặc VRAM không đủ.

In [ ]:
import json
import subprocess

query = [
    "nvidia-smi",
    "--query-gpu=index,name,driver_version,memory.total,memory.free,compute_cap",
    "--format=csv,noheader,nounits",
]
gpu_check = subprocess.run(query, text=True, capture_output=True)
print(gpu_check.stdout or gpu_check.stderr)
if gpu_check.returncode != 0 or not gpu_check.stdout.strip():
    raise RuntimeError("Không tìm thấy CUDA GPU. Hãy bật GPU accelerator trong Kaggle Settings.")

GPU_REPORT = []
for line in gpu_check.stdout.strip().splitlines():
    index, name, driver, total_mib, free_mib, compute_cap = [x.strip() for x in line.split(",")]
    GPU_REPORT.append({
        "index": int(index),
        "name": name,
        "driver_version": driver,
        "memory_total_mib": int(total_mib),
        "memory_free_mib": int(free_mib),
        "compute_capability": compute_cap,
    })

if GPU_REPORT[0]["memory_free_mib"] < 7000:
    raise RuntimeError(f"GPU 0 chỉ còn {GPU_REPORT[0]['memory_free_mib']} MiB; cần ít nhất khoảng 7 GiB trống.")

print(json.dumps(GPU_REPORT, indent=2, ensure_ascii=False))
print("GPU checkpoint: PASS")

## Checkpoint 2 — Repository, input, môi trường và model

Cell đầu cố định commit nguồn, tìm CSV trong `/kaggle/input/`, kiểm tra SHA-256 và xác nhận input không chứa trường gold. Cell sau mới cài Ollama, khởi động server và pull đúng model.

> Nếu có nhiều file cùng tên trong `/kaggle/input/`, đặt `DIAGNOSTIC_CSV_OVERRIDE` thành đường dẫn tuyệt đối của file cần dùng.

In [ ]:
from pathlib import Path
import hashlib
import inspect
import os
import shutil
import sys
import pandas as pd

REPO_URL = "https://github.com/dlxik/vip-girls.git"
SOURCE_BRANCH = "data/cmv-edge-review-20261116"
VERIFIED_COMMIT = "02f97218e5726588bc4c29fbdb61f49bbed28127"
REPO_DIR = Path("/kaggle/working/vip-girls")
DIAGNOSTIC_FILENAME = "model_inputs_human_segmented_5.csv"
DIAGNOSTIC_SHA256 = "A509BA22554AD927C0B5681B916E7EBA8AF98299AD052EA04F2D0280E43474D5"
DIAGNOSTIC_CSV_OVERRIDE = ""  # Chỉ sửa nếu Kaggle tìm thấy nhiều file cùng tên.

MODEL_ID = "qwen2.5:7b"
MODEL_SOURCE_ID = "hf.co/Qwen/Qwen2.5-7B-Instruct-GGUF:Q4_K_M"
MODEL_SPEC = f"ollama:{MODEL_ID}"
NUM_CTX = 8192
TEMPERATURE = 0
SEED = 0
MAX_EXTRACTION_RETRIES = 1
CUDA_DEVICE = "0"
ONE_ITEM_ID = "CMV-038-ARG01"
EXPECTED_IDS = [
    "CMV-017-ARG01",
    "CMV-017-ARG02",
    "CMV-020-ARG01",
    "CMV-020-ARG02",
    "CMV-038-ARG01",
]

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all", "--tags"], check=True)
subprocess.run([
    "git", "-C", str(REPO_DIR), "merge-base", "--is-ancestor",
    VERIFIED_COMMIT, f"origin/{SOURCE_BRANCH}",
], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", VERIFIED_COMMIT], check=True)
actual_commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
assert actual_commit == VERIFIED_COMMIT, (actual_commit, VERIFIED_COMMIT)

if DIAGNOSTIC_CSV_OVERRIDE:
    attached_csv = Path(DIAGNOSTIC_CSV_OVERRIDE)
else:
    matches = sorted(Path("/kaggle/input").rglob(DIAGNOSTIC_FILENAME))
    if len(matches) != 1:
        raise RuntimeError(
            f"Cần đúng 1 file {DIAGNOSTIC_FILENAME} trong /kaggle/input, hiện tìm thấy: {matches}. "
            "Hãy attach dataset hoặc đặt DIAGNOSTIC_CSV_OVERRIDE."
        )
    attached_csv = matches[0]

input_sha256 = hashlib.sha256(attached_csv.read_bytes()).hexdigest().upper()
if input_sha256 != DIAGNOSTIC_SHA256:
    raise RuntimeError(
        f"SHA-256 input không khớp. Nhận {input_sha256}, cần {DIAGNOSTIC_SHA256}. "
        "Hãy upload đúng CSV diagnostic đã chuẩn bị."
    )

WORKING_INPUT = Path("/kaggle/working") / DIAGNOSTIC_FILENAME
if WORKING_INPUT.exists():
    existing_sha = hashlib.sha256(WORKING_INPUT.read_bytes()).hexdigest().upper()
    if existing_sha != DIAGNOSTIC_SHA256:
        raise RuntimeError(f"File đích đã tồn tại nhưng SHA sai: {WORKING_INPUT}")
else:
    shutil.copy2(attached_csv, WORKING_INPUT)

WSDM_DIR = REPO_DIR / "wsdm"
sys.path.insert(0, str(WSDM_DIR))
import approved_detection_experiment as approved_exp
import llm
from llm import OllamaClient
assert hasattr(llm, "extract_pair_with_retry"), (
    "Remote branch chưa chứa extraction retry patch; cần commit/push bản đã review trước khi chạy."
)

inputs = approved_exp._read_csv(WORKING_INPUT)
approved_exp._assert_model_input(inputs)
assert inputs["argument_pair_id"].tolist() == EXPECTED_IDS
assert set(inputs["input_condition"]) == {"human_segmented"}
assert inputs["human_segmentation_used"].astype(str).str.upper().eq("TRUE").all()
forbidden_columns = [
    c for c in inputs.columns
    if any(x in c.lower() for x in ("label", "steering", "evidence", "adjudication", "gold"))
]
assert forbidden_columns == [], forbidden_columns

client_config_probe = OllamaClient(MODEL_ID)
assert client_config_probe.num_ctx == NUM_CTX
complete_source = inspect.getsource(OllamaClient.complete)
assert '"temperature": 0' in complete_source
assert '"seed": 0' in complete_source
assert '"format": "json"' in complete_source

print(f"Repository commit: {actual_commit}")
print(f"Diagnostic input: {attached_csv}")
print(f"SHA-256: {input_sha256}")
print(f"Rows: {len(inputs)}; IDs: {inputs['argument_pair_id'].tolist()}")
print(f"Config cố định: model={MODEL_ID}, num_ctx={NUM_CTX}, temperature={TEMPERATURE}, seed={SEED}")
print("Gold-field isolation: PASS")

In [ ]:
import time
import urllib.request

# Kaggle cần bật Internet cho package install, Ollama installer và model pull.
# Một số Kaggle images không có zstd, trong khi Ollama installer cần nó để giải nén.
if shutil.which("zstd") is None:
    print("Installing missing system dependency: zstd")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "zstd"], check=True)
assert shutil.which("zstd") is not None, "zstd installation failed"

subprocess.run(
    ["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"],
    check=True,
)

os.environ["CUDA_VISIBLE_DEVICES"] = CUDA_DEVICE
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
OLLAMA_LOG = Path("/kaggle/working/ollama_server.log")

def ollama_is_ready():
    try:
        with urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=3) as response:
            return response.status == 200
    except Exception:
        return False

if not ollama_is_ready():
    log_handle = OLLAMA_LOG.open("a", encoding="utf-8")
    OLLAMA_PROCESS = subprocess.Popen(
        ["ollama", "serve"],
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )
    for _ in range(60):
        if ollama_is_ready():
            break
        time.sleep(1)
    else:
        print(OLLAMA_LOG.read_text(encoding="utf-8", errors="replace")[-5000:])
        raise RuntimeError("Ollama server không sẵn sàng sau 60 giây.")

# Q4_K_M chính thức gồm 2 GGUF shards. Ollama HF registry chưa hỗ trợ pull shard,
# nên tải đúng hai file chính thức, ghép bằng llama.cpp rồi import vào Ollama.
HF_BASE = "https://huggingface.co/Qwen/Qwen2.5-7B-Instruct-GGUF/resolve/main"
MODEL_DIR = Path("/kaggle/working/qwen2.5-7b-instruct-q4_k_m")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
SHARD_NAMES = [
    "qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf",
    "qwen2.5-7b-instruct-q4_k_m-00002-of-00002.gguf",
]
for shard_name in SHARD_NAMES:
    shard_path = MODEL_DIR / shard_name
    if not shard_path.exists():
        subprocess.run([
            "curl", "-fL", "--retry", "5", "--retry-all-errors",
            "-o", str(shard_path), f"{HF_BASE}/{shard_name}?download=true",
        ], check=True)
    if shard_path.stat().st_size < 100_000_000:
        raise RuntimeError(f"Shard tải lỗi hoặc quá nhỏ: {shard_path}")

LLAMA_CPP = Path("/kaggle/working/llama.cpp")
if not LLAMA_CPP.exists():
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/ggml-org/llama.cpp.git", str(LLAMA_CPP),
    ], check=True)
subprocess.run([
    "cmake", "-S", str(LLAMA_CPP), "-B", str(LLAMA_CPP / "build"),
    "-DGGML_CUDA=OFF", "-DLLAMA_CURL=OFF",
], check=True)
subprocess.run([
    "cmake", "--build", str(LLAMA_CPP / "build"),
    "--target", "llama-gguf-split", "-j", "2",
], check=True)
GGUF_SPLIT = LLAMA_CPP / "build" / "bin" / "llama-gguf-split"
MERGED_GGUF = MODEL_DIR / "qwen2.5-7b-instruct-q4_k_m.gguf"
if not MERGED_GGUF.exists():
    subprocess.run([
        str(GGUF_SPLIT), "--merge", str(MODEL_DIR / SHARD_NAMES[0]), str(MERGED_GGUF),
    ], check=True)
if MERGED_GGUF.stat().st_size < 4_000_000_000:
    raise RuntimeError(f"Merged GGUF quá nhỏ: {MERGED_GGUF.stat().st_size} bytes")

MODELFILE = MODEL_DIR / "Modelfile"
MODELFILE.write_text(f"FROM {MERGED_GGUF}\n", encoding="utf-8")
subprocess.run(["ollama", "create", MODEL_ID, "-f", str(MODELFILE)], check=True)
subprocess.run(["ollama", "list"], check=True)

with urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=10) as response:
    tags = json.loads(response.read())
model_records = tags.get("models", [])
available_names = {m.get("name") for m in model_records}
assert MODEL_ID in available_names or f"{MODEL_ID}:latest" in available_names, available_names
MODEL_METADATA = next(
    m for m in model_records if m.get("name") in {MODEL_ID, f"{MODEL_ID}:latest"}
)

print("Resolved model metadata:", json.dumps(MODEL_METADATA, indent=2, ensure_ascii=False))
print("Environment/model checkpoint: PASS")
print("Chưa chạy extraction. Hãy xem GPU/model output rồi mới sang checkpoint 3.")

## Checkpoint 3 — Inference một item

Chạy riêng `CMV-038-ARG01`, một ca grounded uptake rõ. Cell gọi retry có kiểm soát, validator và `ProposedScorer` từ repository. Raw response và lỗi của mọi attempt đều được giữ; extraction lỗi không bị biến thành score 0.

In [ ]:
from llm import EXTRACTION_KEYS, extract_pair_with_retry, make_client, validate_extraction
from steering import ProposedScorer, pair_key

one_row = inputs.loc[inputs["argument_pair_id"].eq(ONE_ITEM_ID)].iloc[0]
one_item = approved_exp._edge_item(one_row)

print("=" * 100)
print(f"ONE-ITEM DIAGNOSTIC: {ONE_ITEM_ID}")
print("\n[A — human_segmented]\n", one_row["A_text"])
print("\n[B — human_segmented]\n", one_row["B_text"])
print("=" * 100)

one_client = make_client(
    MODEL_SPEC,
    "/kaggle/working/one_item_qwen2_5_7b_cache.jsonl",
)
one_extraction, one_attempts = extract_pair_with_retry(
    one_client, one_item, "A1", max_retries=1,
)
print("Attempt audit:")
print(json.dumps(one_attempts, indent=2, ensure_ascii=False))

ONE_ITEM_DIAGNOSTIC = {"argument_pair_id": ONE_ITEM_ID}
if one_extraction is None:
    ONE_ITEM_DIAGNOSTIC["status"] = "unparseable_json"
    ONE_ITEM_DIAGNOSTIC["attempts"] = one_attempts
    print("EXTRACTION FAILURE: unparseable JSON; không tạo score.")
elif not one_extraction["validation"]["valid"]:
    ONE_ITEM_DIAGNOSTIC["status"] = "invalid_extraction"
    ONE_ITEM_DIAGNOSTIC["errors"] = one_extraction["validation"]["errors"]
    ONE_ITEM_DIAGNOSTIC["attempts"] = one_attempts
    print("EXTRACTION FAILURE:", one_extraction["validation"]["errors"])
    print("Không tạo score.")
else:
    clean_extraction = {k: one_extraction[k] for k in EXTRACTION_KEYS}
    revalidation_errors = validate_extraction(clean_extraction, one_item, "A1")
    assert revalidation_errors == [], revalidation_errors
    mapping = {pair_key(one_item.candidates["A1"].id, one_item.B.id): one_extraction}
    score_rows = []
    for variant in ProposedScorer.VARIANTS:
        scorer = ProposedScorer(mapping, variant)
        explanation = scorer.explain(one_item, "A1")
        score_rows.append({
            "method": scorer.name,
            "score": explanation["S"],
            **explanation["components"],
        })
    ONE_ITEM_DIAGNOSTIC["status"] = "valid"
    ONE_ITEM_DIAGNOSTIC["extraction"] = clean_extraction
    ONE_ITEM_DIAGNOSTIC["attempts"] = one_attempts
    print("\nExtracted evidence:")
    print("span_A:", one_extraction["span_A"])
    print("span_B:", one_extraction["span_B"])
    print("frame_A:", one_extraction["frame_A"])
    print("frame_B:", one_extraction["frame_B"])
    display(pd.DataFrame(score_rows))

print("\nHãy kiểm tra thủ công spans, frame, G/U/D và score trước khi chạy cả 5 item.")

## Checkpoint 4 — Inference cả 5 item

Cell này gọi nguyên trạng CLI `approved_detection_experiment.py extract-proposed`. Output directory phải chưa tồn tại để tránh ghi đè. Không truyền gold file và không gọi `evaluate-proposed`.

In [ ]:
OUTPUT_DIR = Path("/kaggle/working/experiment1_diagnostic5_qwen2_5_7b_retry1")
if OUTPUT_DIR.exists():
    raise FileExistsError(
        f"{OUTPUT_DIR} đã tồn tại. Adapter chủ ý không ghi đè; "
        "hãy đổi OUTPUT_DIR cho một run mới hoặc khởi động Kaggle session mới."
    )

command = [
    sys.executable,
    str(WSDM_DIR / "approved_detection_experiment.py"),
    "extract-proposed",
    "--inputs", str(WORKING_INPUT),
    "--model", MODEL_SPEC,
    "--out", str(OUTPUT_DIR),
    "--max-extraction-retries", "1",
]
print("Running command:", " ".join(command))
subprocess.run(command, cwd=str(REPO_DIR), check=True)

expected_outputs = [
    "llm_cache.jsonl",
    "extractions.jsonl",
    "extraction_errors.jsonl",
    "extraction_attempts.jsonl",
    "proposed_predictions.csv",
    "run_manifest.json",
]
for name in expected_outputs:
    path = OUTPUT_DIR / name
    print(f"{name}: exists={path.exists()}, bytes={path.stat().st_size if path.exists() else 'NA'}")
    assert path.exists(), path

print("Five-item extraction checkpoint: COMPLETE")
print("Chưa tính accuracy hoặc metric khoa học.")

## Checkpoint 5 — Validate, inspect và export

Cell cuối đọc output, chạy lại validator trên JSON/spans, in A/B gốc cùng evidence và G/U/D, ghi execution manifest bổ sung và tạo ZIP trong `/kaggle/working/`. Nó không đọc gold và không tính accuracy.

In [ ]:
from datetime import datetime, timezone
from pandas.errors import EmptyDataError

def read_jsonl(path):
    records = []
    with Path(path).open(encoding="utf-8") as handle:
        for line_no, line in enumerate(handle, 1):
            if line.strip():
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError as exc:
                    raise RuntimeError(f"Invalid JSONL at {path}:{line_no}: {exc}") from exc
    return records

extractions = read_jsonl(OUTPUT_DIR / "extractions.jsonl")
errors = read_jsonl(OUTPUT_DIR / "extraction_errors.jsonl")
attempts = read_jsonl(OUTPUT_DIR / "extraction_attempts.jsonl")
try:
    predictions = pd.read_csv(OUTPUT_DIR / "proposed_predictions.csv", encoding="utf-8-sig")
except EmptyDataError:
    predictions = pd.DataFrame()

input_by_id = {row.argument_pair_id: row for row in inputs.itertuples(index=False)}
valid_by_id = {record["argument_pair_id"]: record for record in extractions}
errors_by_id = {record["argument_pair_id"]: record for record in errors}
attempts_by_id = {item_id: [a for a in attempts if a["argument_pair_id"] == item_id] for item_id in EXPECTED_IDS}
assert all(attempts_by_id.values())
assert all(1 <= len(records) <= 2 for records in attempts_by_id.values())
assert not (set(valid_by_id) & set(errors_by_id))
assert set(valid_by_id) | set(errors_by_id) == set(EXPECTED_IDS)

for argument_pair_id, extraction in valid_by_id.items():
    row = input_by_id[argument_pair_id]
    item = approved_exp._edge_item(row)
    clean = {k: extraction[k] for k in EXTRACTION_KEYS}
    validation_errors = validate_extraction(clean, item, "A1")
    if validation_errors:
        raise RuntimeError(f"Revalidation failed for {argument_pair_id}: {validation_errors}")

for argument_pair_id in EXPECTED_IDS:
    row = input_by_id[argument_pair_id]
    print("\n" + "=" * 110)
    print("argument_pair_id:", argument_pair_id)
    print("input_condition:", row.input_condition)
    print("\n[A — original model input]\n" + row.A_text)
    print("\n[B — original model input]\n" + row.B_text)
    print("\n[ATTEMPT AUDIT]")
    for attempt in attempts_by_id[argument_pair_id]:
        print(json.dumps(attempt, indent=2, ensure_ascii=False))
    if argument_pair_id in errors_by_id:
        print("\nEXTRACTION FAILURE (không gán score 0):")
        print(json.dumps(errors_by_id[argument_pair_id], indent=2, ensure_ascii=False))
        continue

    extraction = valid_by_id[argument_pair_id]
    print("\n[EXTRACTED EVIDENCE]")
    print("span_A:", extraction["span_A"])
    print("span_B:", extraction["span_B"])
    print("span_pair_ok:", extraction["span_pair_ok"])
    print("specific_reference:", extraction["specific_reference"])
    print("frame_A:", extraction["frame_A"])
    print("frame_B:", extraction["frame_B"])
    print("uptake_level/type:", extraction["uptake_level"], "/", extraction["uptake_type"])
    print("d_aspect/d_lens:", extraction["d_aspect"], "/", extraction["d_lens"])

    item_predictions = predictions.loc[
        predictions["argument_pair_id"].eq(argument_pair_id),
        ["method", "score", "Temporal", "G", "U", "U_raw", "D", "d_aspect", "d_lens"],
    ]
    print("\n[SCORES — công thức/variants hiện có]")
    display(item_predictions.reset_index(drop=True))

adapter_manifest = json.loads((OUTPUT_DIR / "run_manifest.json").read_text(encoding="utf-8"))
assert adapter_manifest["model"] == MODEL_SPEC
assert adapter_manifest["n_inputs"] == 5
assert adapter_manifest["formula"] == "S(A→B|C) = Temporal × G × U × D"

KAGGLE_MANIFEST = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "purpose": "five-item deliberately selected diagnostic; no scientific accuracy calculation",
    "repository_url": REPO_URL,
    "source_branch": SOURCE_BRANCH,
    "verified_commit": VERIFIED_COMMIT,
    "resolved_commit": actual_commit,
    "input_condition": "human_segmented",
    "input_file": str(WORKING_INPUT),
    "input_sha256": input_sha256,
    "n_items": 5,
    "gold_loaded": False,
    "accuracy_calculated": False,
    "model_backend": MODEL_SPEC,
    "model_id": MODEL_ID,
    "model_source_id": MODEL_SOURCE_ID,
    "ollama_model_metadata": globals().get("MODEL_METADATA", {}),
    "num_ctx": NUM_CTX,
    "temperature": TEMPERATURE,
    "seed": SEED,
    "cuda_visible_devices": CUDA_DEVICE,
    "gpu_report": globals().get("GPU_REPORT", []),
    "n_valid_extractions": len(extractions),
    "n_extraction_failures": len(errors),
    "n_extraction_attempts": len(attempts),
    "max_extraction_retries": 1,
    "formula": "S(A→B|C) = Temporal × G × U × D",
    "adapter_run_manifest": adapter_manifest,
}
(OUTPUT_DIR / "kaggle_run_manifest.json").write_text(
    json.dumps(KAGGLE_MANIFEST, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

ZIP_PATH = Path(shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR))
print("\nEXPORT SUMMARY")
for path in sorted(OUTPUT_DIR.iterdir()):
    if path.is_file():
        print(f"- {path} ({path.stat().st_size} bytes)")
print(f"- ZIP: {ZIP_PATH} ({ZIP_PATH.stat().st_size} bytes)")
print(f"Coverage kỹ thuật: {len(extractions)}/5 valid; {len(errors)}/5 failures")
print("Không tính AUROC, AP, accuracy hoặc threshold metric trên subset chẩn đoán này.")

## Rủi ro còn lại khi diễn giải

- Đây là điều kiện **human-segmented**; model nhận argument spans do con người phân đoạn.
- Approved source không có historical context hoặc raw timestamps. Adapter dùng context rỗng và ordinal `A=0, B=1` sau khi temporal order đã được xác minh.
- Tất cả source pairs được biểu diễn là direct reply; direct reply một mình vẫn không đủ vì gate còn yêu cầu `span_pair_ok`.
- Qwen2.5:7b là model nhỏ/quantized; JSON có thể hợp schema nhưng frame hoặc evidence vẫn cần researcher kiểm tra định tính.
- Năm item được chọn bằng gold để bao phủ failure modes. Không dùng kết quả này làm accuracy hoặc paper-level performance estimate.
